In [1]:
from __future__ import (absolute_import, division, print_function,
                        unicode_literals)

import datetime  # For datetime objects
import os.path  # To manage paths
import sys  # To find out the script name (in argv[0])

# Import the backtrader platform
import backtrader as bt
import backtrader.indicators as btind
import backtrader.feeds as btfeeds

In [8]:
symbols = ['AAPL', 'NVDA', 'AMZN']

datafeeds = []
for symbol in symbols:
    data = btfeeds.GenericCSVData(
        dataname=f'{symbol.lower()}_data.csv',
        
        # Date/time parameters
        datetime=1,  # timestamp column
        dtformat='%Y-%m-%d %H:%M:%S%z',  # matches your format with timezone
        
        # OHLCV columns
        open=2,
        high=3,
        low=4,
        close=5,
        volume=6,
        
        # Extra columns you don't need for backtrader
        openinterest=-1,  # not present in your data
        
        # Optional: date range filtering
        fromdate=datetime.datetime(2016, 1, 1),
        todate=datetime.datetime(2017, 1, 1),
    )

    # Set a name for the data feed
    data.plotinfo.plotname = symbol 
    data._name = symbol

    datafeeds.append(data)

In [ ]:
"""
Notes:
self.params == self.p
self.data.lines.close[0] == self.data.close[0] == self.data.l.close[0]
self.data0 == self.datas[0]
"""

'\nNotes:\nself.params == self.p\nself.data.lines.close[0] == self.data.close[0] == self.data.l.close[0]\nself.data0 == self.datas[0]\n'

In [12]:
# Create a Stratey
class TestStrategy(bt.Strategy):
    params = dict(short_period=15, long_period=60)
    def log(self, txt, dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def _log_positions(self):
        self.log('--- PORTFOLIO STATUS ---')
        self.log('Total Portfolio Value: %.2f' % self.broker.getvalue())
        
        # Iterate through every data feed loaded in Cerebro
        found_holdings = False
        for data in self.datas:
            # Get the position size for this specific data feed
            pos_size = self.getposition(data).size
            
            # Only print if we actually hold shares (not 0)
            if pos_size != 0:
                found_holdings = True
                # Use data._name to get the Ticker (if set)
                name = data._name if data._name else 'Unknown Asset'
                self.log(f'  > HOLDING: {name} | Size: {pos_size:.2f} | Total Value: {pos_size * data.close[0]:.2f}')
        
        if not found_holdings:
            self.log('  > No active positions')
        self.log('------------------------------------------------')

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        if order.status in [order.Completed]:
            buy_or_sell = 'BUY' if order.isbuy() else 'SELL'
            self.log(f'{buy_or_sell} EXECUTED {order.data._name}, Price: {order.executed.price:.2f}, Cost: {order.executed.value:.2f}')
            self.bar_executed = len(self)
            self._log_positions()
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('Order Canceled/Margin/Rejected for %s' % order.data._name)

    def notify_trade(self, trade):
        if not trade.isclosed:
            return
        self.log('OPERATION PROFIT, GROSS %.2f, NET %.2f' % (trade.pnl, trade.pnlcomm))
        

    def __init__(self):
        self.crossovers = {}
        for data in self.datas:
            short_moving_average = btind.ExponentialMovingAverage(data, period=self.p.short_period)
            long_moving_average = btind.SimpleMovingAverage(data, period=self.p.long_period)
            crossover = bt.indicators.CrossOver(short_moving_average, long_moving_average)
            self.crossovers[data._name] = crossover

    def next(self):
        for data in self.datas:
            crossover = self.crossovers[data._name]
            if crossover > 0:
                self.buy(data=data)
                self.log(f'BUY, {data._name}, {data.close[0]:.2f}')
            elif crossover < 0:
                self.sell(data=data)
                self.log(f'SELL, {data._name}, {data.close[0]:.2f}')


cerebro = bt.Cerebro()
# ENABLE CHEAT-ON-CLOSE
cerebro.broker.set_coc(True)
for feed in datafeeds:
    cerebro.adddata(feed)
cerebro.addstrategy(TestStrategy)
cerebro.broker.setcash(100000.0)
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
cerebro.run()
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

Starting Portfolio Value: 100000.00
2016-05-02, SELL, AAPL, 93.64
2016-05-03, SELL EXECUTED AAPL, Price: 93.64, Cost: -93.64
2016-05-03, --- PORTFOLIO STATUS ---
2016-05-03, Total Portfolio Value: 99998.46
2016-05-03,   > HOLDING: AAPL | Size: -1.00 | Total Value: -95.18
2016-05-03, ------------------------------------------------
2016-07-15, BUY, AAPL, 98.78
2016-07-18, BUY EXECUTED AAPL, Price: 98.78, Cost: -93.64
2016-07-18, --- PORTFOLIO STATUS ---
2016-07-18, Total Portfolio Value: 99994.86
2016-07-18,   > No active positions
2016-07-18, ------------------------------------------------
2016-07-18, OPERATION PROFIT, GROSS -5.14, NET -5.14
2016-11-07, SELL, AMZN, 784.93
2016-11-08, SELL EXECUTED AMZN, Price: 784.93, Cost: -784.93
2016-11-08, --- PORTFOLIO STATUS ---
2016-11-08, Total Portfolio Value: 99992.04
2016-11-08,   > HOLDING: AMZN | Size: -1.00 | Total Value: -787.75
2016-11-08, ------------------------------------------------
2016-11-10, SELL, AAPL, 107.79
2016-11-11, SELL 

In [5]:
cerebro.plot(iplot=False)

[[<Figure size 2340x1580 with 9 Axes>]]